# Debt Settlement Direct Mail Campaign Model - Sample Workflow

This notebook uses **synthetic data only**. It demonstrates a modeling workflow for debt settlement direct mail campaign selection using credit bureau-style attributes and historical client performance metrics.

Business goals:

- Improve response rate
- Improve enrollment conversion
- Estimate enrolled dollar amount
- Select prospects by expected profit / ROI
- Create practical campaign groups: `do_not_mail`, `test_cell`, `mail_standard`, `mail_priority`

When real data arrives, we will replace the synthetic data section with actual credit bureau + campaign history inputs.

In [1]:
from __future__ import annotations

import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

## 1. Create Synthetic Campaign Data

The fake dataset includes:

- Credit bureau-style features: FICO, unsecured debt, utilization, delinquencies, inquiries, income estimate, DTI
- Campaign/client features: client id, creative channel, prior response rate, prior enrollment rate, prior average enrolled dollars
- Outcomes: response, enrollment, enrolled dollar amount

In [2]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))


def make_synthetic_campaign_data(n: int = 50_000, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    state = rng.choice(["CA", "TX", "FL", "NY", "GA", "IL", "AZ", "NC"], size=n)
    client_id = rng.choice(["client_a", "client_b", "client_c", "client_d"], size=n)
    channel = rng.choice(["letter_a", "letter_b", "letter_c"], size=n, p=[0.45, 0.35, 0.20])

    fico = np.clip(rng.normal(610, 75, n), 430, 820)
    unsecured_debt = rng.lognormal(mean=10.1, sigma=0.55, size=n)
    revolving_utilization = np.clip(rng.beta(4, 2, n), 0, 1)
    delinquency_30d_count = rng.poisson(0.8, n)
    recent_inquiry_count = rng.poisson(2.0, n)
    monthly_income_est = rng.lognormal(mean=10.45, sigma=0.45, size=n)
    debt_to_income = unsecured_debt / (monthly_income_est * 12)
    credit_card_trade_count = rng.poisson(6, n) + 1

    client_perf = pd.DataFrame(
        {
            "client_id": ["client_a", "client_b", "client_c", "client_d"],
            "client_prior_resp_rate": [0.038, 0.052, 0.031, 0.045],
            "client_prior_enroll_rate": [0.36, 0.31, 0.42, 0.34],
            "client_prior_avg_enrolled_dollars": [18_500, 15_200, 21_000, 16_800],
        }
    )

    df = pd.DataFrame(
        {
            "prospect_id": np.arange(1, n + 1),
            "client_id": client_id,
            "state": state,
            "creative_channel": channel,
            "fico": fico.round(0),
            "unsecured_debt": unsecured_debt.round(2),
            "revolving_utilization": revolving_utilization.round(4),
            "delinquency_30d_count": delinquency_30d_count,
            "recent_inquiry_count": recent_inquiry_count,
            "monthly_income_est": monthly_income_est.round(2),
            "debt_to_income": debt_to_income.round(4),
            "credit_card_trade_count": credit_card_trade_count,
        }
    ).merge(client_perf, on="client_id", how="left")

    response_logit = (
        -4.7
        + 0.45 * (df["client_prior_resp_rate"] / 0.05)
        + 0.65 * df["revolving_utilization"]
        + 0.35 * np.log1p(df["unsecured_debt"]) / 10
        + 0.08 * df["recent_inquiry_count"]
        - 0.003 * (df["fico"] - 600)
        + df["creative_channel"].map({"letter_a": 0.05, "letter_b": 0.18, "letter_c": -0.08})
    )
    response_prob = np.clip(sigmoid(response_logit), 0.005, 0.45)
    df["response"] = rng.binomial(1, response_prob)

    enroll_logit = (
        -1.35
        + 1.9 * df["client_prior_enroll_rate"]
        + 1.2 * df["debt_to_income"]
        + 0.25 * df["delinquency_30d_count"]
        - 0.000018 * df["monthly_income_est"]
        + df["state"].map({"CA": 0.08, "TX": 0.03, "FL": 0.06, "NY": -0.05}).fillna(0)
    )
    enroll_prob_given_response = np.clip(sigmoid(enroll_logit), 0.03, 0.75)
    df["enrollment"] = np.where(df["response"].eq(1), rng.binomial(1, enroll_prob_given_response), 0)

    amount_noise = rng.normal(0, 2_500, n)
    expected_amount = (
        0.52 * df["unsecured_debt"]
        + 0.18 * df["client_prior_avg_enrolled_dollars"]
        + 4_000 * df["debt_to_income"]
        - 7 * (df["fico"] - 600)
        + amount_noise
    )
    df["enrolled_dollars"] = np.where(df["enrollment"].eq(1), np.clip(expected_amount, 2_000, 75_000), 0).round(2)

    return df


df = make_synthetic_campaign_data()
df.head()

,prospect_id,client_id,state,creative_channel,fico,unsecured_debt,revolving_utilization,delinquency_30d_count,recent_inquiry_count,monthly_income_est,debt_to_income,credit_card_trade_count,client_prior_resp_rate,client_prior_enroll_rate,client_prior_avg_enrolled_dollars,response,enrollment,enrolled_dollars
0,1,client_a,CA,letter_c,690.0,47987.96,0.8244,0,3,49451.49,0.0809,5,0.038,0.36,18500,0,0,0.0
1,2,client_d,AZ,letter_a,512.0,18044.06,0.4237,0,1,25735.20,0.0584,7,0.045,0.34,16800,0,0,0.0
2,3,client_c,IL,letter_b,652.0,23165.84,0.7559,0,2,37343.93,0.0517,7,0.031,0.42,21000,0,0,0.0
3,4,client_d,NY,letter_a,611.0,35152.17,0.7663,1,2,22141.82,0.1323,7,0.045,0.34,16800,0,0,0.0
4,5,client_b,NY,letter_b,788.0,13938.48,0.3621,0,3,40737.74,0.0285,6,0.052,0.31,15200,0,0,0.0


## 2. Quick Outcome Checks

In [3]:
outcome_summary = pd.Series(
    {
        "prospects": len(df),
        "response_rate": df["response"].mean(),
        "enroll_rate_overall": df["enrollment"].mean(),
        "enroll_rate_given_response": df.loc[df["response"].eq(1), "enrollment"].mean(),
        "avg_enrolled_dollars_given_enrolled": df.loc[df["enrollment"].eq(1), "enrolled_dollars"].mean(),
    }
)
outcome_summary

prospects                              50000.000000
response_rate                              0.035260
enroll_rate_overall                        0.009540
enroll_rate_given_response                 0.270562
avg_enrolled_dollars_given_enrolled    19175.080084
dtype: float64

## 3. Feature Setup

For real data, this is where we will map bureau fields and client history fields into model-ready features.

In [4]:
numeric_features = [
    "fico",
    "unsecured_debt",
    "revolving_utilization",
    "delinquency_30d_count",
    "recent_inquiry_count",
    "monthly_income_est",
    "debt_to_income",
    "credit_card_trade_count",
    "client_prior_resp_rate",
    "client_prior_enroll_rate",
    "client_prior_avg_enrolled_dollars",
]
categorical_features = ["client_id", "state", "creative_channel"]
feature_cols = numeric_features + categorical_features


def build_preprocessor(numeric_features: list[str], categorical_features: list[str]) -> ColumnTransformer:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])

## 4. Train Three Models

We model the funnel explicitly:

1. Probability of response
2. Probability of enrollment conditional on response
3. Enrolled dollar amount conditional on enrollment

Then we combine them into expected enrolled dollars and expected profit.

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=df["response"],
)

preprocessor = build_preprocessor(numeric_features, categorical_features)

response_model = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(max_iter=180, learning_rate=0.045, random_state=RANDOM_SEED)),
])
response_model.fit(train_df[feature_cols], train_df["response"])

responders_train = train_df[train_df["response"].eq(1)].copy()
responders_test = test_df[test_df["response"].eq(1)].copy()

enrollment_model = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(max_iter=160, learning_rate=0.05, random_state=RANDOM_SEED)),
])
enrollment_model.fit(responders_train[feature_cols], responders_train["enrollment"])

enrolled_train = train_df[train_df["enrollment"].eq(1)].copy()
amount_model = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingRegressor(max_iter=180, learning_rate=0.045, random_state=RANDOM_SEED)),
])
amount_model.fit(enrolled_train[feature_cols], enrolled_train["enrolled_dollars"])

print("Models trained.")

Models trained.


## 5. Score Prospects and Calculate ROI

In [6]:
scored = test_df.copy()
scored["p_response"] = response_model.predict_proba(scored[feature_cols])[:, 1]
scored["p_enroll_given_response"] = enrollment_model.predict_proba(scored[feature_cols])[:, 1]
scored["predicted_enrolled_dollars_if_enrolled"] = amount_model.predict(scored[feature_cols]).clip(0)
scored["expected_enrolled_dollars"] = (
    scored["p_response"]
    * scored["p_enroll_given_response"]
    * scored["predicted_enrolled_dollars_if_enrolled"]
)

mail_cost = 0.72
settlement_fee_rate = 0.19
gross_margin_rate = 0.55

scored["expected_revenue"] = scored["expected_enrolled_dollars"] * settlement_fee_rate
scored["predicted_profit"] = scored["expected_revenue"] * gross_margin_rate - mail_cost
scored["predicted_roi"] = scored["predicted_profit"] / mail_cost

scored["selection_group"] = pd.cut(
    scored["predicted_profit"],
    bins=[-np.inf, 5, 15, 35, np.inf],
    labels=["do_not_mail", "test_cell", "mail_standard", "mail_priority"],
)

scored[["prospect_id", "p_response", "p_enroll_given_response", "expected_enrolled_dollars", "predicted_profit", "predicted_roi", "selection_group"]].head()

,prospect_id,p_response,p_enroll_given_response,expected_enrolled_dollars,predicted_profit,predicted_roi,selection_group
7485,7486,0.028078,0.507054,281.331581,28.679150,39.832153,mail_standard
11064,11065,0.025126,0.644707,408.335175,41.951026,58.265314,mail_priority
7096,7097,0.060811,0.010794,8.337078,0.151225,0.210034,do_not_mail
40246,40247,0.027109,0.131589,30.548481,2.472316,3.433773,do_not_mail
47038,47039,0.023904,0.130182,38.999132,3.355409,4.660291,do_not_mail


## 6. Model Metrics

In [7]:
response_auc = roc_auc_score(scored["response"], scored["p_response"])
response_ap = average_precision_score(scored["response"], scored["p_response"])
enrollment_auc = roc_auc_score(
    responders_test["enrollment"],
    enrollment_model.predict_proba(responders_test[feature_cols])[:, 1],
)

pd.Series({
    "response_auc": response_auc,
    "response_average_precision": response_ap,
    "enrollment_given_response_auc": enrollment_auc,
})

response_auc                     0.566156
response_average_precision       0.042588
enrollment_given_response_auc    0.580478
dtype: float64

## 7. Selection Group Counts

In [8]:
scored["selection_group"].value_counts(dropna=False).sort_index()

selection_group
do_not_mail      5026
test_cell        5503
mail_standard    3248
mail_priority    1223
Name: count, dtype: int64

## 8. Decile Summary by Predicted Profit

This is the main selection view. In real campaign validation, we would compare response, enrollment, enrolled dollars, cost, revenue, and ROI by decile.

In [9]:
def decile_summary(df: pd.DataFrame, score_col: str) -> pd.DataFrame:
    scored_df = df.sort_values(score_col, ascending=False).copy()
    scored_df["selection_decile"] = pd.qcut(
        scored_df[score_col].rank(method="first", ascending=False),
        q=10,
        labels=np.arange(1, 11),
    ).astype(int)

    return (
        scored_df.groupby("selection_decile", as_index=False)
        .agg(
            prospects=("prospect_id", "count"),
            actual_response_rate=("response", "mean"),
            actual_enroll_rate=("enrollment", "mean"),
            actual_avg_enrolled_dollars=("enrolled_dollars", "mean"),
            predicted_profit_sum=("predicted_profit", "sum"),
            actual_enrolled_dollars_sum=("enrolled_dollars", "sum"),
        )
        .sort_values("selection_decile")
    )

decile_summary(scored, "predicted_profit")

,selection_decile,prospects,actual_response_rate,actual_enroll_rate,actual_avg_enrolled_dollars,predicted_profit_sum,actual_enrolled_dollars_sum
0,1,1500,0.037333,0.011333,294.367347,89843.966204,441551.02
1,2,1500,0.036667,0.012667,250.905993,37513.520683,376358.99
2,3,1500,0.036667,0.012667,224.214773,26271.169487,336322.16
3,4,1500,0.036000,0.011333,174.236967,19320.337020,261355.45
4,5,1500,0.032667,0.011333,210.321433,14493.553942,315482.15
5,6,1500,0.041333,0.012000,217.623967,10893.343279,326435.95
6,7,1500,0.036667,0.006000,100.540940,7900.015594,150811.41
7,8,1500,0.035333,0.008000,151.084427,5436.301423,226626.64
8,9,1500,0.028667,0.007333,104.046893,3242.396034,156070.34
9,10,1500,0.031333,0.006000,91.525453,1064.249343,137288.18


## 9. Top Prospects for Mailing

In [10]:
output_cols = [
    "prospect_id",
    "client_id",
    "state",
    "p_response",
    "p_enroll_given_response",
    "expected_enrolled_dollars",
    "predicted_profit",
    "predicted_roi",
    "selection_group",
]

scored.sort_values("predicted_profit", ascending=False)[output_cols].head(20)

,prospect_id,client_id,state,p_response,p_enroll_given_response,expected_enrolled_dollars,predicted_profit,predicted_roi,selection_group
8247,8248,client_a,NY,0.070357,0.974072,3292.721366,343.369383,476.901921,mail_priority
41193,41194,client_a,NC,0.064210,0.945317,3081.518044,321.298636,446.248105,mail_priority
18357,18358,client_d,AZ,0.066019,0.893045,2838.325667,295.885032,410.951434,mail_priority
21557,21558,client_c,IL,0.085410,0.651907,2660.702289,277.323389,385.171374,mail_priority
38906,38907,client_d,AZ,0.083200,0.673808,2619.285544,272.995339,379.160194,mail_priority
20714,20715,client_a,FL,0.057624,0.910617,2527.926567,263.448326,365.900453,mail_priority
24431,24432,client_c,CA,0.071702,0.743163,2526.610320,263.310778,365.709414,mail_priority
21823,21824,client_d,GA,0.051605,0.969285,2407.679684,250.882527,348.447954,mail_priority
3500,3501,client_c,CA,0.058930,0.859524,2378.058409,247.787104,344.148755,mail_priority
2633,2634,client_d,GA,0.048866,0.934754,2140.814915,222.995159,309.715498,mail_priority
